In [6]:
from pathlib import Path
import pandas as pd

In [11]:
import s3fs
import configparser

In [27]:
from flows_integration import task_combine_NASC_to_dataframe
from flows_biology import add_stratum

In [1]:
path_vm_local = "/media/volume/shimada_202506_volume/integration" # path to local integrated data
path_NASC_files = "agr230002-bucket01/prefect_sh2506_test/acoustics/NASC" # path to cloud acoustic data
cred_file =  "/home/exouser/.config/rclone/rclone.conf"
file_NASC_all = "NASC_all.csv"
file_stratum_mean = "stratum_mean.csv"
file_NASC_all_grid = "NASC_all_griddify.geojson"
num_NASC_reprocess = 1

In [2]:
if num_NASC_reprocess < 1:
    num_NASC_reprocess = 1
    raise Warning("num_NASC_reprocess must be at least 1, setting it to 1.")

In [7]:
file_NASC_all = Path(path_vm_local) / file_NASC_all
df_NASC_all = (
    pd.read_csv(file_NASC_all, index_col=0)
    if file_NASC_all.exists() else pd.DataFrame()
)
NASC_processed = (
    sorted(df_NASC_all["filename"].unique().tolist())
    if not df_NASC_all.empty else []
)

In [8]:
NASC_to_reprocess = NASC_processed[-num_NASC_reprocess:]
NASC_processed = NASC_processed[:-num_NASC_reprocess]

In [10]:
NASC_processed

['NASC_20250613T114000.zarr',
 'NASC_20250613T122000.zarr',
 'NASC_20250613T130000.zarr',
 'NASC_20250613T134000.zarr',
 'NASC_20250613T142000.zarr',
 'NASC_20250613T150000.zarr',
 'NASC_20250613T154000.zarr',
 'NASC_20250613T162000.zarr',
 'NASC_20250613T170000.zarr',
 'NASC_20250613T174000.zarr',
 'NASC_20250613T182000.zarr',
 'NASC_20250613T190000.zarr',
 'NASC_20250613T194000.zarr',
 'NASC_20250613T202000.zarr',
 'NASC_20250613T210000.zarr',
 'NASC_20250613T214000.zarr',
 'NASC_20250613T222000.zarr',
 'NASC_20250613T230000.zarr',
 'NASC_20250613T234000.zarr',
 'NASC_20250614T002000.zarr',
 'NASC_20250614T114000.zarr',
 'NASC_20250614T122000.zarr',
 'NASC_20250614T130000.zarr',
 'NASC_20250614T134000.zarr',
 'NASC_20250614T142000.zarr',
 'NASC_20250614T150000.zarr',
 'NASC_20250614T154000.zarr',
 'NASC_20250614T162000.zarr',
 'NASC_20250614T170000.zarr',
 'NASC_20250614T174000.zarr',
 'NASC_20250614T182000.zarr',
 'NASC_20250614T190000.zarr',
 'NASC_20250614T194000.zarr',
 'NASC_202

In [12]:
config = configparser.ConfigParser()
config.read(cred_file)
fs = s3fs.S3FileSystem(
    key=config["osn_sdsc_hake"]["access_key_id"],
    secret=config["osn_sdsc_hake"]["secret_access_key"],
    client_kwargs={"endpoint_url": config["osn_sdsc_hake"]["endpoint"]},
)

In [13]:
NASC_all = fs.glob(f"{path_NASC_files}/*.zarr")
NASC_all = sorted([Path(f).name for f in NASC_all])

In [14]:
NASC_to_process = sorted(list(set(NASC_all).difference(set(NASC_processed))))

In [16]:
NASC_to_process_original = NASC_to_process.copy()

In [18]:
NASC_to_process = NASC_to_process[:10]

In [19]:
files_to_process = ""
for nascf in NASC_to_process:
    files_to_process += f"- {nascf}\n"
print(f"Files to process:\n{files_to_process}")

Files to process:
- NASC_20250720T130000.zarr
- NASC_20250720T134000.zarr
- NASC_20250720T142000.zarr
- NASC_20250720T150000.zarr
- NASC_20250720T154000.zarr
- NASC_20250720T162000.zarr
- NASC_20250720T170000.zarr
- NASC_20250720T174000.zarr
- NASC_20250720T182000.zarr
- NASC_20250720T190000.zarr



In [21]:
df_NASC, errors = task_combine_NASC_to_dataframe(fs, path_NASC_files, NASC_to_process)

17:31:55.966 | INFO    | Task run 'task_combine_NASC_to_dataframe' - Finished in state Completed()


In [22]:
df_NASC_all_original = df_NASC_all.copy()

In [23]:
df_NASC_all

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,stratum,sigma_bs_mean,weight_mean,number_density,biomass_density
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,1.0,0.000099,0.109571,0.000000e+00,0.000000e+00
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,1.0,0.000099,0.109571,0.000000e+00,0.000000e+00
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,1.0,0.000099,0.109571,0.000000e+00,0.000000e+00
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500000256,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,1.0,0.000099,0.109571,0.000000e+00,0.000000e+00
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,1.0,0.000099,0.109571,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...
3833,2816.327141,38000.0,48.028884,-125.312925,2025-09-06 23:34:07.500000256,WBT 400143-15 ES38B_ES,NASC_20250906T230000.zarr,5.0,0.000370,0.735468,7.606052e+06,5.594010e+06
3834,177.281093,38000.0,48.032465,-125.317152,2025-09-06 23:39:27.500000,WBT 400143-15 ES38B_ES,NASC_20250906T230000.zarr,5.0,0.000370,0.735468,4.787828e+05,3.521296e+05
3835,0.000000,38000.0,48.036689,-125.321011,2025-09-06 23:45:05,WBT 400143-15 ES38B_ES,NASC_20250906T234000.zarr,5.0,0.000370,0.735468,0.000000e+00,0.000000e+00
3836,0.000000,38000.0,48.044053,-125.327108,2025-09-06 23:56:57.499999744,WBT 400143-15 ES38B_ES,NASC_20250906T234000.zarr,5.0,0.000370,0.735468,0.000000e+00,0.000000e+00


In [24]:
df_NASC

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename
0,3183.881197,38000.0,38.176756,-123.229316,2025-07-20 13:02:05.000000512,WBT 400143-15 ES38B_ES,NASC_20250720T130000.zarr
1,996.166001,38000.0,38.182718,-123.222194,2025-07-20 13:06:07.499999744,WBT 400143-15 ES38B_ES,NASC_20250720T130000.zarr
2,1200.797872,38000.0,38.188882,-123.214978,2025-07-20 13:09:45.000000000,WBT 400143-15 ES38B_ES,NASC_20250720T130000.zarr
3,1288.434694,38000.0,38.195053,-123.207816,2025-07-20 13:13:07.500000000,WBT 400143-15 ES38B_ES,NASC_20250720T130000.zarr
4,0.000000,38000.0,38.201271,-123.200657,2025-07-20 13:16:24.999999744,WBT 400143-15 ES38B_ES,NASC_20250720T130000.zarr
...,...,...,...,...,...,...,...
92,786.779317,38000.0,38.272731,-123.506731,2025-07-20 18:54:12.500000000,WBT 400143-15 ES38B_ES,NASC_20250720T182000.zarr
93,0.000000,38000.0,38.271806,-123.515276,2025-07-20 19:05:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr
94,0.000000,38000.0,38.266289,-123.524393,2025-07-20 19:15:20.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr
95,0.000000,38000.0,38.259474,-123.526211,2025-07-20 19:26:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr


In [25]:
df_NASC_all = pd.concat([df_NASC_all, df_NASC], ignore_index=True)
df_NASC_all

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,stratum,sigma_bs_mean,weight_mean,number_density,biomass_density
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,1.0,0.000099,0.109571,0.0,0.0
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,1.0,0.000099,0.109571,0.0,0.0
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,1.0,0.000099,0.109571,0.0,0.0
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500000256,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,1.0,0.000099,0.109571,0.0,0.0
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,1.0,0.000099,0.109571,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
3930,786.779317,38000.0,38.272731,-123.506731,2025-07-20 18:54:12.500000,WBT 400143-15 ES38B_ES,NASC_20250720T182000.zarr,NaN,NaN,NaN,NaN,NaN
3931,0.000000,38000.0,38.271806,-123.515276,2025-07-20 19:05:00,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,NaN,NaN,NaN
3932,0.000000,38000.0,38.266289,-123.524393,2025-07-20 19:15:20.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,NaN,NaN,NaN
3933,0.000000,38000.0,38.259474,-123.526211,2025-07-20 19:26:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,NaN,NaN,NaN


In [26]:
df_stratum = pd.read_csv(Path(path_vm_local) / file_stratum_mean, index_col=0)

In [34]:
df_NASC_all = df_NASC_all.drop(
    ["stratum", "sigma_bs_mean", "weight_mean"],  # drop previous stratification
    axis=1
)

In [36]:
df_NASC_all = add_stratum(df_NASC_all, df_stratum)

In [37]:
df_NASC_all

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,number_density,biomass_density,stratum
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500000256,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.0,0.0,1
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...
3930,786.779317,38000.0,38.272731,-123.506731,2025-07-20 18:54:12.500000,WBT 400143-15 ES38B_ES,NASC_20250720T182000.zarr,NaN,NaN,2
3931,0.000000,38000.0,38.271806,-123.515276,2025-07-20 19:05:00,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2
3932,0.000000,38000.0,38.266289,-123.524393,2025-07-20 19:15:20.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2
3933,0.000000,38000.0,38.259474,-123.526211,2025-07-20 19:26:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2


In [38]:
df_NASC_all["stratum"] = df_NASC_all["stratum"].astype("Int64")  # convert from category to int

In [39]:
df_NASC_all

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,number_density,biomass_density,stratum
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500000256,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.0,0.0,1
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...
3930,786.779317,38000.0,38.272731,-123.506731,2025-07-20 18:54:12.500000,WBT 400143-15 ES38B_ES,NASC_20250720T182000.zarr,NaN,NaN,2
3931,0.000000,38000.0,38.271806,-123.515276,2025-07-20 19:05:00,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2
3932,0.000000,38000.0,38.266289,-123.524393,2025-07-20 19:15:20.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2
3933,0.000000,38000.0,38.259474,-123.526211,2025-07-20 19:26:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2


In [40]:
df_stratum

,stratum,latitude_northern_limit,stratum_name,sigma_bs_mean,weight_mean
0,1,36.0000,Conception,0.000099,0.109571
1,2,40.5000,Monterey,0.000155,0.199259
2,3,43.0000,Eureka,0.000211,0.325607
3,4,45.7667,South Columbia,0.000261,0.437096
4,5,48.5000,North Columbia - US/CAN border,0.000370,0.735468
5,6,55.0000,Canada,NaN,NaN


In [41]:
df_NASC_all.merge(
            df_stratum[["stratum", "sigma_bs_mean", "weight_mean"]],
            on="stratum",  # merge on stratum 
            how="left"
        )

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,number_density,biomass_density,stratum,sigma_bs_mean,weight_mean
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1,0.000099,0.109571
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1,0.000099,0.109571
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.0,0.0,1,0.000099,0.109571
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500000256,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.0,0.0,1,0.000099,0.109571
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000000000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.0,0.0,1,0.000099,0.109571
...,...,...,...,...,...,...,...,...,...,...,...,...
3930,786.779317,38000.0,38.272731,-123.506731,2025-07-20 18:54:12.500000,WBT 400143-15 ES38B_ES,NASC_20250720T182000.zarr,NaN,NaN,2,0.000155,0.199259
3931,0.000000,38000.0,38.271806,-123.515276,2025-07-20 19:05:00,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2,0.000155,0.199259
3932,0.000000,38000.0,38.266289,-123.524393,2025-07-20 19:15:20.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2,0.000155,0.199259
3933,0.000000,38000.0,38.259474,-123.526211,2025-07-20 19:26:45.000000256,WBT 400143-15 ES38B_ES,NASC_20250720T190000.zarr,NaN,NaN,2,0.000155,0.199259


In [74]:
import geopandas as gpd

In [75]:
path_vm_local = "/media/volume/shimada_202506_volume/integration" # path to local integrated data
file_NASC_all_grid = "NASC_all_griddify.geojson"
file_stratum_mean = "stratum_mean.csv"

In [76]:
file_NASC_all_grid = Path(path_vm_local) / file_NASC_all_grid
file_stratum_mean = Path(path_vm_local) / file_stratum_mean

In [77]:
# Load griddified NASC and grid cells
gdf_NASC = gpd.read_file(file_NASC_all_grid)

# Load stratum means
df_stratum = pd.read_csv(file_stratum_mean, index_col=0)

In [78]:
from grid import create_grid_from_bounds
from core import GRID_PARAMS

In [79]:
# Create gdf_grid_cells
gdf_grid_cells, _, _ = create_grid_from_bounds(
    bounds=GRID_PARAMS["bounds"], 
    resolution=GRID_PARAMS["resolution"],
    projection=GRID_PARAMS["projection"],
    coastline_resolution="10m",
    area_threshold=5
)
gdf_grid_cells.set_index(["grid_x", "grid_y"], inplace=True)

In [80]:
gdf_grid_cells

,,geometry,area
grid_x,grid_y,,
38,1,"POLYGON ((-117.25452 32.76713, -117.25388 32.7...",491.943240
37,1,"POLYGON ((-117.4894 32.78924, -117.49448 32.75...",625.000000
36,1,"POLYGON ((-118.44992 32.81882, -118.43837 32.8...",610.460377
35,1,"POLYGON ((-118.51361 32.88037, -118.5086 32.87...",620.304729
34,1,"POLYGON ((-118.94753 32.9164, -118.96635 32.75...",625.000000
...,...,...,...
5,53,"POLYGON ((-135.24446 54.8394, -135.18116 54.42...",625.000000
6,53,"POLYGON ((-134.52703 54.87429, -134.47091 54.4...",625.000000
5,54,"POLYGON ((-135.24446 54.8394, -135.25 54.83909...",625.000000


In [81]:
gdf_NASC

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,number_density,biomass_density,stratum,sigma_bs_mean,weight_mean,utm_x,utm_y,grid_x,grid_y,geometry
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1.0,0.000099,0.109571,1.557471e+06,3.671434e+06,36.0,1.0,POINT (1557470.775 3671434.318)
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1.0,0.000099,0.109571,1.556477e+06,3.671556e+06,36.0,1.0,POINT (1556477.094 3671556.433)
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1.0,0.000099,0.109571,1.555817e+06,3.671433e+06,36.0,1.0,POINT (1555816.658 3671432.7)
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.000000e+00,0.000000,1.0,0.000099,0.109571,1.555585e+06,3.671015e+06,36.0,1.0,POINT (1555584.688 3671014.944)
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.000000e+00,0.000000,1.0,0.000099,0.109571,1.555960e+06,3.670065e+06,36.0,1.0,POINT (1555959.612 3670064.718)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7979,0.000000,38000.0,48.454628,-126.255252,2025-09-08 18:02:52.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,0.000000e+00,0.000000,5.0,0.000370,0.735468,7.029312e+05,5.370471e+06,18.0,38.0,POINT (702931.244 5370471.444)
7980,284.715550,38000.0,48.454514,-126.242787,2025-09-08 18:07:00.000,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,7.689310e+05,565524.366702,5.0,0.000370,0.735468,7.038532e+05,5.370492e+06,18.0,38.0,POINT (703853.169 5370491.879)
7981,375.032086,38000.0,48.454230,-126.230429,2025-09-08 18:11:07.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,1.012849e+06,744918.157640,5.0,0.000370,0.735468,7.047679e+05,5.370493e+06,18.0,38.0,POINT (704767.913 5370493.399)
7982,146.724656,38000.0,48.454051,-126.217837,2025-09-08 18:15:17.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,3.962591e+05,291436.025656,5.0,0.000370,0.735468,7.056995e+05,5.370507e+06,18.0,38.0,POINT (705699.527 5370507.286)


In [82]:
# Drop sigma_bs_mean and weight_mean to avoid merge conflicts
gdf_NASC = gdf_NASC.drop(
    ["sigma_bs_mean", "weight_mean"],
    axis=1, errors='ignore'
)

In [83]:
gdf_NASC

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,number_density,biomass_density,stratum,utm_x,utm_y,grid_x,grid_y,geometry
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1.0,1.557471e+06,3.671434e+06,36.0,1.0,POINT (1557470.775 3671434.318)
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1.0,1.556477e+06,3.671556e+06,36.0,1.0,POINT (1556477.094 3671556.433)
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1.0,1.555817e+06,3.671433e+06,36.0,1.0,POINT (1555816.658 3671432.7)
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.000000e+00,0.000000,1.0,1.555585e+06,3.671015e+06,36.0,1.0,POINT (1555584.688 3671014.944)
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.000000e+00,0.000000,1.0,1.555960e+06,3.670065e+06,36.0,1.0,POINT (1555959.612 3670064.718)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7979,0.000000,38000.0,48.454628,-126.255252,2025-09-08 18:02:52.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,0.000000e+00,0.000000,5.0,7.029312e+05,5.370471e+06,18.0,38.0,POINT (702931.244 5370471.444)
7980,284.715550,38000.0,48.454514,-126.242787,2025-09-08 18:07:00.000,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,7.689310e+05,565524.366702,5.0,7.038532e+05,5.370492e+06,18.0,38.0,POINT (703853.169 5370491.879)
7981,375.032086,38000.0,48.454230,-126.230429,2025-09-08 18:11:07.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,1.012849e+06,744918.157640,5.0,7.047679e+05,5.370493e+06,18.0,38.0,POINT (704767.913 5370493.399)
7982,146.724656,38000.0,48.454051,-126.217837,2025-09-08 18:15:17.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,3.962591e+05,291436.025656,5.0,7.056995e+05,5.370507e+06,18.0,38.0,POINT (705699.527 5370507.286)


In [84]:
gdf_NASC["stratum"] = gdf_NASC["stratum"].astype("Int64")  # convert from category to int

In [85]:
gdf_NASC

,NASC,frequency_nominal,latitude,longitude,ping_time,channel,filename,number_density,biomass_density,stratum,utm_x,utm_y,grid_x,grid_y,geometry
0,0.000000,38000.0,32.672269,-117.751696,2025-06-13 11:57:37.500,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1,1.557471e+06,3.671434e+06,36.0,1.0,POINT (1557470.775 3671434.318)
1,0.000000,38000.0,32.674293,-117.761954,2025-06-13 12:08:45.000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1,1.556477e+06,3.671556e+06,36.0,1.0,POINT (1556477.094 3671556.433)
2,0.000000,38000.0,32.673825,-117.769001,2025-06-13 12:17:00.000,WBT 400143-15 ES38B_ES,NASC_20250613T114000.zarr,0.000000e+00,0.000000,1,1.555817e+06,3.671433e+06,36.0,1.0,POINT (1555816.658 3671432.7)
3,0.000000,38000.0,32.670349,-117.771896,2025-06-13 12:22:32.500,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.000000e+00,0.000000,1,1.555585e+06,3.671015e+06,36.0,1.0,POINT (1555584.688 3671014.944)
4,0.000000,38000.0,32.661586,-117.769040,2025-06-13 12:27:15.000,WBT 400143-15 ES38B_ES,NASC_20250613T122000.zarr,0.000000e+00,0.000000,1,1.555960e+06,3.670065e+06,36.0,1.0,POINT (1555959.612 3670064.718)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7979,0.000000,38000.0,48.454628,-126.255252,2025-09-08 18:02:52.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,0.000000e+00,0.000000,5,7.029312e+05,5.370471e+06,18.0,38.0,POINT (702931.244 5370471.444)
7980,284.715550,38000.0,48.454514,-126.242787,2025-09-08 18:07:00.000,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,7.689310e+05,565524.366702,5,7.038532e+05,5.370492e+06,18.0,38.0,POINT (703853.169 5370491.879)
7981,375.032086,38000.0,48.454230,-126.230429,2025-09-08 18:11:07.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,1.012849e+06,744918.157640,5,7.047679e+05,5.370493e+06,18.0,38.0,POINT (704767.913 5370493.399)
7982,146.724656,38000.0,48.454051,-126.217837,2025-09-08 18:15:17.500,WBT 400143-15 ES38B_ES,NASC_20250908T174000.zarr,3.962591e+05,291436.025656,5,7.056995e+05,5.370507e+06,18.0,38.0,POINT (705699.527 5370507.286)


In [88]:
gdf_NASC = gdf_NASC.merge(
    df_stratum[["stratum", "sigma_bs_mean", "weight_mean"]],
    on="stratum",  # merge on stratum 
    how="left"
)

In [89]:
gdf_NASC["number_density"] = gdf_NASC["NASC"] / gdf_NASC["sigma_bs_mean"]